In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os
from IPython.display import SVG, display

In [ ]:
!pwd

In [ ]:
# 文件路径
data_file = r"./result/sam_atten_cams.csv"
data = pd.read_csv(data_file)
save_file = r"./sam_homo_atten_vis"

# 如果保存路径不存在，创建路径
if not os.path.exists(save_file):
    os.makedirs(save_file)

In [ ]:
# 计算全局范围（vmin 和 vmax）
def calculate_global_range(data, model_columns):
    all_colors = []
    for column in model_columns:
        for colors in data[column]:
            all_colors.extend(np.fromstring(colors.strip('[]'), sep=' '))
    return min(all_colors), max(all_colors)

In [ ]:
# 设置模型列
model_columns = ["MODEL_ALL"]
vmin, vmax = calculate_global_range(data, model_columns)
norm = plt.Normalize(vmin=vmin, vmax=vmax)
cmap = cm.get_cmap('YlGn')

In [ ]:
def draw(smiles, colors, save_path=None, norm=None, cmap=None, highlight_radius=0.5):
    """
    绘制分子结构图，并调整原子高亮圈的大小。
    
    Parameters:
    - smiles (str): 分子 SMILES。
    - colors (list or np.ndarray): 每个原子的着色值。
    - save_path (str, optional): 保存路径。
    - norm (matplotlib.colors.Normalize, optional): 颜色归一化。
    - cmap (matplotlib.colors.Colormap, optional): 颜色映射表。
    - highlight_radius (float, optional): 原子高亮圈的半径，默认值为 0.5。
    """
    # 构建分子对象
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        raise ValueError("无效的 SMILES 表达式")
    rdMolDraw2D.PrepareMolForDrawing(mol)
    
    # 检查输入的颜色值
    if isinstance(colors, str):
        colors = np.fromstring(colors.strip('[]'), sep=' ')
    elif not isinstance(colors, (list, np.ndarray)):
        raise ValueError("`colors` 应为列表或数组格式")
    
    if len(colors) != mol.GetNumAtoms():
        raise ValueError("`colors` 的长度应与分子中原子数量相同")

    # 颜色归一化和映射
    if norm is None:
        norm = plt.Normalize(vmin=min(colors), vmax=max(colors))
    if cmap is None:
        cmap = cm.get_cmap("PuBu")
    
    atom_colors = {}
    for i, value in enumerate(colors):
        rgba = cmap(norm(value))
        atom_colors[i] = (rgba[0], rgba[1], rgba[2])
    
    # 配置 RDKit 绘图选项
    drawer = rdMolDraw2D.MolDraw2DSVG(600, 600)
    options = drawer.drawOptions()
    options.bondLineWidth = 4  # 调整键线宽
    options.atomLabelFontFace = "Helvetica"
    options.fixedFontSize = 26
    options.useBWAtomPalette()  # 强制原子标签为黑色
    options.fixedBondLength = 30
    options.highlightRadius = highlight_radius  # 设置高亮圈的半径
    options.boldAtomLabels = True
    options.rotate = 280
    # 绘制分子并高亮原子
    drawer.DrawMolecule(
        mol, 
        highlightAtoms=list(atom_colors.keys()), 
        highlightAtomColors=atom_colors, 
        highlightBonds=[]
    )
    drawer.FinishDrawing()
    svg = drawer.GetDrawingText().replace("svg:", "")
    
    # 创建 colorbar
    fig, ax = plt.subplots(figsize=(0.2, 4))
    cbar = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), cax=ax, orientation="vertical")
    cbar.set_label("Color Scale", rotation=270, labelpad=20)
    
    # 调整 colorbar 边框和刻度线粗细
    cbar.ax.tick_params(width=1.5)  # 调整刻度线粗细
    for spine in cbar.ax.spines.values():
        spine.set_linewidth(1.5)  # 调整边框粗细
    
    # 保存 colorbar
    if save_path:
        colorbar_path = save_path.replace(".svg", "_colorbar.png")
        plt.savefig(colorbar_path, bbox_inches="tight", dpi=300)
    plt.show()
    # 保存文件
    if save_path:
        # 如果 save_path 是目录，则为每个分子生成唯一文件名
        if os.path.isdir(save_path):
            file_name = f"{smiles.replace('/', '_')}.svg"  # 使用分子的前10个字符生成文件名
            save_path = os.path.join(save_path, file_name)
        with open(save_path, "w") as f:
            f.write(svg)
    display(SVG(svg))

In [ ]:
# 按 SMILES 顺序绘制并保存分子图
for i in range(data.shape[0]):
    smiles = data.loc[i, "SMILES"]
    colors = data.loc[i, "MODEL_ALL"]
    draw(smiles, colors, save_path=save_file, norm=norm, cmap=cmap, highlight_radius=0.5)